# 02 — GTFS processing (pipeline)

Merge configured agencies (MTS + NCTD), **prefix IDs** to avoid cross-feed collisions, **clip** to `bbox` using *trips that touch the bbox* (full trip geometry retained for routing), validate referential integrity, then write:

- **`data/processed/gtfs/sd_merged_bbox/`** — standard GTFS `*.txt` tables (UTF-8)
- **`data/processed/gtfs/sd_merged_bbox.zip`** — same files, for **r5py** `TransportNetwork`
- **`artifacts/tables/pipeline/pipeline__02_*.csv`** — summary stats

**Previous:** [`01_data_exploration.ipynb`](01_data_exploration.ipynb) · **Next:** [`03_accessibility_computation.ipynb`](03_accessibility_computation.ipynb)


In [ ]:
from __future__ import annotations

from pathlib import Path

import yaml

# Merge defaults + city config (same pattern as notebooks/eda/02)


def find_repo_root() -> Path:
    start = Path.cwd().resolve()
    for d in [start, *start.parents]:
        if (d / "configs" / "san_diego.yaml").exists():
            return d
    raise FileNotFoundError("Could not find configs/san_diego.yaml.")


REPO_ROOT = find_repo_root()
with open(REPO_ROOT / "configs" / "defaults.yaml", encoding="utf-8") as f:
    CONFIG: dict = yaml.safe_load(f)
with open(REPO_ROOT / "configs" / "san_diego.yaml", encoding="utf-8") as f:
    CONFIG.update(yaml.safe_load(f))

bbox = CONFIG["bbox"]
min_lon, min_lat, max_lon, max_lat = bbox
agencies_cfg = CONFIG.get("gtfs_agencies") or []


In [ ]:
import zipfile
from datetime import date

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

GTFS_TABLES = (
    "agency",
    "routes",
    "trips",
    "stops",
    "stop_times",
    "calendar",
    "calendar_dates",
    "shapes",
    "transfers",
    "feed_info",
)


def load_gtfs_feed(extracted_dir: Path) -> dict[str, pd.DataFrame | None]:
    out: dict[str, pd.DataFrame | None] = {}
    for name in GTFS_TABLES:
        p = extracted_dir / f"{name}.txt"
        if not p.exists():
            out[name] = None
            continue
        out[name] = pd.read_csv(p, dtype=str, encoding="utf-8", low_memory=False)
    return out


def _pfx(s: pd.Series, prefix: str) -> pd.Series:
    return prefix + s.astype(str)


def prefix_feed(feed: dict[str, pd.DataFrame | None], aid: str) -> dict[str, pd.DataFrame | None]:
    """Make stop/trip/route/service/shape IDs unique across agencies."""
    prefix = f"{aid}:"
    out = {}
    for k, df in feed.items():
        if df is None or df.empty:
            out[k] = df
            continue
        d = df.copy()
        if k == "agency" and "agency_id" in d.columns:
            d["agency_id"] = _pfx(d["agency_id"], prefix)
        if k == "routes" and "route_id" in d.columns:
            d["route_id"] = _pfx(d["route_id"], prefix)
            if "agency_id" in d.columns:
                d["agency_id"] = _pfx(d["agency_id"], prefix)
            else:
                # GTFS allows routes without agency_id for single-agency feeds.
                # Synthesise it so R5 can resolve the agency reference.
                _agency_df = feed.get("agency")
                if _agency_df is not None and len(_agency_df) == 1 and "agency_id" in _agency_df.columns:
                    d["agency_id"] = prefix + str(_agency_df["agency_id"].iloc[0])
        if k == "trips":
            for col in ("route_id", "trip_id", "service_id"):
                if col in d.columns:
                    d[col] = _pfx(d[col], prefix)
            if "shape_id" in d.columns:
                m = d["shape_id"].notna() & (d["shape_id"].astype(str).str.strip() != "")
                d.loc[m, "shape_id"] = prefix + d.loc[m, "shape_id"].astype(str)
        if k == "stops" and "stop_id" in d.columns:
            d["stop_id"] = _pfx(d["stop_id"], prefix)
            # Also prefix parent_station so R5 can resolve station refs
            if "parent_station" in d.columns:
                _m = d["parent_station"].notna() & (d["parent_station"].astype(str).str.strip() != "")
                d.loc[_m, "parent_station"] = prefix + d.loc[_m, "parent_station"].astype(str)
        if k == "stop_times":
            d["trip_id"] = _pfx(d["trip_id"], prefix)
            d["stop_id"] = _pfx(d["stop_id"], prefix)
        if k == "shapes" and "shape_id" in d.columns:
            d["shape_id"] = _pfx(d["shape_id"], prefix)
        if k == "calendar" and "service_id" in d.columns:
            d["service_id"] = _pfx(d["service_id"], prefix)
        if k == "calendar_dates" and "service_id" in d.columns:
            d["service_id"] = _pfx(d["service_id"], prefix)
        if k == "transfers":
            for col in ("from_stop_id", "to_stop_id"):
                if col in d.columns:
                    d[col] = _pfx(d[col], prefix)
        out[k] = d
    return out


def concat_feeds(feeds: list[dict[str, pd.DataFrame | None]]) -> dict[str, pd.DataFrame | None]:
    keys = set()
    for f in feeds:
        keys |= set(f.keys())
    merged: dict[str, pd.DataFrame | None] = {}
    for name in sorted(keys):
        parts = [f[name] for f in feeds if f.get(name) is not None and not f[name].empty]
        merged[name] = pd.concat(parts, ignore_index=True) if parts else None
    return merged


def stops_in_bbox(stops: pd.DataFrame) -> pd.Series:
    lat = stops["stop_lat"].astype(float)
    lon = stops["stop_lon"].astype(float)
    return (lon >= min_lon) & (lon <= max_lon) & (lat >= min_lat) & (lat <= max_lat)


def clip_merged_to_bbox(F: dict[str, pd.DataFrame | None]) -> dict[str, pd.DataFrame | None]:
    """Keep trips that have at least one stop in bbox; retain all stop_times rows for those trips."""
    stops = F.get("stops")
    stop_times = F.get("stop_times")
    trips = F.get("trips")
    if stops is None or stops.empty or stop_times is None or stop_times.empty or trips is None or trips.empty:
        raise ValueError("Merged feed missing stops, stop_times, or trips.")

    inside = stops_in_bbox(stops)
    seed = set(stops.loc[inside, "stop_id"].astype(str))
    st = stop_times.copy()
    st["trip_id"] = st["trip_id"].astype(str)
    st["stop_id"] = st["stop_id"].astype(str)
    trip_keep = set(st.loc[st["stop_id"].isin(seed), "trip_id"].unique())
    st_kept = st[st["trip_id"].isin(trip_keep)].copy()
    stop_keep = set(st_kept["stop_id"].unique())
    stops_kept = stops[stops["stop_id"].astype(str).isin(stop_keep)].copy()
    trips_kept = trips[trips["trip_id"].astype(str).isin(trip_keep)].copy()

    route_keep = set(trips_kept["route_id"].astype(str).unique()) if "route_id" in trips_kept.columns else set()
    svc_keep = set(trips_kept["service_id"].astype(str).unique()) if "service_id" in trips_kept.columns else set()
    shape_keep = (
        set(trips_kept["shape_id"].dropna().astype(str).unique())
        if "shape_id" in trips_kept.columns
        else set()
    )

    out: dict[str, pd.DataFrame | None] = {}
    out["stops"] = stops_kept
    out["stop_times"] = st_kept
    out["trips"] = trips_kept

    r = F.get("routes")
    out["routes"] = r[r["route_id"].astype(str).isin(route_keep)].copy() if r is not None and not r.empty else None

    a = F.get("agency")
    if a is not None and not a.empty and "agency_id" in a.columns and out["routes"] is not None and not out["routes"].empty:
        used_agency = set(out["routes"]["agency_id"].astype(str).unique())
        out["agency"] = a[a["agency_id"].astype(str).isin(used_agency)].copy()
    else:
        out["agency"] = a

    cal = F.get("calendar")
    out["calendar"] = cal[cal["service_id"].astype(str).isin(svc_keep)].copy() if cal is not None and not cal.empty else None

    cd = F.get("calendar_dates")
    out["calendar_dates"] = (
        cd[cd["service_id"].astype(str).isin(svc_keep)].copy() if cd is not None and not cd.empty else None
    )

    sh = F.get("shapes")
    if sh is not None and not sh.empty and shape_keep:
        out["shapes"] = sh[sh["shape_id"].astype(str).isin(shape_keep)].copy()
    else:
        out["shapes"] = sh

    tr = F.get("transfers")
    if tr is not None and not tr.empty:
        valid = stop_keep
        tr2 = tr[tr["from_stop_id"].astype(str).isin(valid) & tr["to_stop_id"].astype(str).isin(valid)].copy()
        out["transfers"] = tr2 if not tr2.empty else None
    else:
        out["transfers"] = None

    fi = F.get("feed_info")
    out["feed_info"] = fi

    return out


def validate_feed(F: dict[str, pd.DataFrame | None]) -> dict[str, int | float]:
    """Lightweight integrity checks (orphan rows)."""
    trips = F.get("trips")
    st = F.get("stop_times")
    rpt: dict[str, int | float] = {}
    if trips is None or st is None:
        return rpt
    tid_t = set(trips["trip_id"].astype(str))
    tid_s = set(st["trip_id"].astype(str))
    rpt["orphan_stop_time_trips"] = int(len(tid_s - tid_t))
    rpt["trips_without_stop_times"] = int(len(tid_t - tid_s))
    return rpt


# ── Load + merge ───────────────────────────────────────────────────────────────
feeds_raw: list[dict[str, pd.DataFrame | None]] = []
for ag in agencies_cfg:
    aid = ag["id"]
    ex = REPO_ROOT / ag["raw_path"].strip().rstrip("/\\").replace("\\", "/") / "extracted"
    if not ex.is_dir():
        raise FileNotFoundError(f"Missing extracted GTFS for {aid}: {ex}")
    feeds_raw.append(prefix_feed(load_gtfs_feed(ex), aid))

merged = concat_feeds(feeds_raw)
display(Markdown("### Merged (pre-clip) row counts"))
display(pd.DataFrame([{k: len(v) if v is not None else 0 for k, v in merged.items()}]).T.rename(columns={0: "rows"}))


In [ ]:
clipped = clip_merged_to_bbox(merged)
issues = validate_feed(clipped)
display(Markdown("### Clipped to bbox — trip expansion"))
display(pd.DataFrame([{k: len(v) if v is not None else 0 for k, v in clipped.items()}]).T.rename(columns={0: "rows"}))
display(Markdown("### Validation"))
display(pd.DataFrame([issues]).T.rename(columns={0: "value"}))


In [ ]:
# Write GTFS text tables + zip for r5py
OUT_DIR = REPO_ROOT / "data" / "processed" / "gtfs" / "sd_merged_bbox"
OUT_DIR.mkdir(parents=True, exist_ok=True)
TODAY = date.today().isoformat()
ART_TAB = REPO_ROOT / "artifacts" / "tables" / "pipeline"
ART_TAB.mkdir(parents=True, exist_ok=True)

written = []
for name in GTFS_TABLES:
    df = clipped.get(name)
    if df is None or df.empty:
        continue
    p = OUT_DIR / f"{name}.txt"
    df.to_csv(p, index=False, encoding="utf-8")
    written.append(name)

ZIP_PATH = REPO_ROOT / "data" / "processed" / "gtfs" / "sd_merged_bbox.zip"
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for name in written:
        p = OUT_DIR / f"{name}.txt"
        zf.write(p, arcname=f"{name}.txt")

summary_rows = {
    "bbox": str(bbox),
    "n_tables_written": len(written),
    "out_dir": OUT_DIR.relative_to(REPO_ROOT).as_posix(),
    "zip": ZIP_PATH.relative_to(REPO_ROOT).as_posix(),
    "n_stops": len(clipped["stops"]) if clipped.get("stops") is not None else 0,
    "n_trips": len(clipped["trips"]) if clipped.get("trips") is not None else 0,
    "n_stop_times": len(clipped["stop_times"]) if clipped.get("stop_times") is not None else 0,
}
for k, v in issues.items():
    summary_rows[k] = v
sum_df = pd.DataFrame([summary_rows])
sum_path = ART_TAB / f"pipeline__02_gtfs_clip_summary__{TODAY}.csv"
sum_df.to_csv(sum_path, index=False)
display(Markdown(f"### Wrote **{len(written)}** tables + `{ZIP_PATH.name}`"))
display(sum_df.T.rename(columns={0: "value"}))


In [ ]:
# Quick map: clipped stops vs bbox (dpi=200 → artifacts/figures)
st = clipped.get("stops")
if st is not None and not st.empty:
    fig, ax = plt.subplots(figsize=(7, 7))
    ax.scatter(
        st["stop_lon"].astype(float),
        st["stop_lat"].astype(float),
        s=2,
        alpha=0.35,
        c="steelblue",
    )
    ax.plot([min_lon, max_lon, max_lon, min_lon, min_lon], [min_lat, min_lat, max_lat, max_lat, min_lat], "r--", lw=1.5, label="bbox")
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel("lon")
    ax.set_ylabel("lat")
    ax.set_title("Clipped GTFS stops (merged MTS+NCTD)")
    ax.legend(loc="upper right")
    plt.tight_layout()
    fig_path = REPO_ROOT / "artifacts" / "figures" / f"pipeline__02_gtfs_stops_clip__{TODAY}.png"
    fig_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(fig_path, dpi=200, bbox_inches="tight")
    plt.close(fig)
    display(Markdown(f"Saved `{fig_path.relative_to(REPO_ROOT)}`"))
else:
    display(Markdown("_No stops to plot._"))
